# Machiavellian Council of Agents

### Agents:

* **Architect**: Work Ethic & Fortune
* **Oracle**: Luck & Fate
* **Survivor**: Random Pain & Hardship
* **Prince**: Orchestrator synthesizing Love & Destiny


In [ ]:
# !pip install openai-agents loguru nest_asyncio

In [ ]:
import asyncio
from loguru import logger
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from agents import Agent, Runner, ModelSettings, OpenAIChatCompletionsModel
from agents import set_default_openai_client, set_default_openai_api
from agents import set_tracing_disabled



In [ ]:
from agents import RunHooks
from agents import AgentHooks
import json
class RawToolArgumentsLoggerRunHook(RunHooks):
    async def on_tool_start(self, context, agent, tool):
        logger.info(f"{agent.name} is calling the {tool.name} \n{json.loads(context.tool_arguments)['input']}")
    async def on_tool_end(self, context, agent, tool, result):
            logger.info(f"{agent.name} received an answer from {tool.name} \n{result}")
class RawToolArgumentsLoggerAgentHook(AgentHooks):
    async def on_tool_start(self, context, agent, tool):
        logger.info(f"{agent.name} is calling {tool.name} \n{json.loads(context.tool_arguments)['input']}")


# Building the Great Outage scenario


In [ ]:
from pathlib import Path

GREAT_OUTAGE_PREFIX = "./great_outage/knowledge"

GREAT_OUTAGE_ENCYCLOPEDIA = Path(f"{GREAT_OUTAGE_PREFIX}/encyclopedia.md").read_text(encoding="utf-8")
GREAT_OUTAGE_ALPHA_CONTEXT = Path(f"{GREAT_OUTAGE_PREFIX}/alpha-context.md").read_text(encoding="utf-8")
GREAT_OUTAGE_DELTA_CONTEXT = Path(f"{GREAT_OUTAGE_PREFIX}/delta-context.md").read_text(encoding="utf-8")
GREAT_OUTAGE_EVENTS = Path(f"{GREAT_OUTAGE_PREFIX}/events.md").read_text(encoding="utf-8")
GREAT_OUTAGE_ALPHA_PROMPT = Path(f"{GREAT_OUTAGE_PREFIX}/alpha-prompt.md").read_text(encoding="utf-8")
GREAT_OUTAGE_DELTA_PROMPT = Path(f"{GREAT_OUTAGE_PREFIX}/delta-prompt.md").read_text(encoding="utf-8")

GREAT_OUTAGE_ALPHA_BASELINE = f"""
# [Encyclopedia]
{GREAT_OUTAGE_ENCYCLOPEDIA}

# [Alpha baseline at Turn 0]
{GREAT_OUTAGE_ALPHA_CONTEXT}

# [Events feed]
{GREAT_OUTAGE_EVENTS}
"""

GREAT_OUTAGE_ALPHA_INSTRUCTIONS = f"""
{GREAT_OUTAGE_ALPHA_BASELINE}

# [Next steps prompt]
{GREAT_OUTAGE_ALPHA_PROMPT}
"""


# Model definition


In [ ]:
BASE_URL = "YOUR_OPENAI_COMPATIBLE_BASE_URL"
API_KEY = "YOUR_API_KEY"

openai_client = AsyncOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)

set_default_openai_client(openai_client)
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

def make_model(model_name: str) -> OpenAIChatCompletionsModel:
    return OpenAIChatCompletionsModel(
        model=model_name,
        openai_client=openai_client,
    )


# Model parameters


In [ ]:
ARCHITECT_MODEL = "YOUR_ARCHITECT_MODEL"
ORACLE_MODEL = "YOUR_ORACLE_MODEL"
SURVIVOR_MODEL = "YOUR_SURVIVOR_MODEL"
PRINCE_MODEL = "YOUR_PRINCE_MODEL"

ARCHITECT_TEMPERATURE = 1.0
ORACLE_TEMPERATURE = 1.0
SURVIVOR_TEMPERATURE = 1.0
PRINCE_TEMPERATURE = 1.0


# Architect agent


In [ ]:
ARCHITECT_INSTRUCTIONS = f"""
{GREAT_OUTAGE_ALPHA_BASELINE}
You are Agent A: The Architect.

Domain:
- Work Ethic & Fortune.
- Political labor, institutional design, long-term preparation.
- Strategic patience and accumulated advantage.
- Preparation before crises arise.

Perspective:
- Analyze what can be controlled through disciplined agency.
- Treat planning as risk management, not prediction.
- Emphasize institutional resilience, training, governance, and readiness.

Output style:
- Concise, structured, practical.
- Speak from the perspective of preparation and institutional design.
"""

architect = Agent(
    name="The Architect 📐",
    model=make_model(ARCHITECT_MODEL),
    handoff_description=(
        "Specialist in Work Ethic & Fortune: preparation, institutions, "
        "strategic patience, and controllable variables."
    ),
    instructions=ARCHITECT_INSTRUCTIONS,
    hooks=RawToolArgumentsLoggerAgentHook(),
    model_settings=ModelSettings(
        temperature=ARCHITECT_TEMPERATURE,
        log_agent_calls=True,
        log_agent_handoffs=True,
    ),
)


# Oracle Agent


In [ ]:
ORACLE_INSTRUCTIONS = f"""
{GREAT_OUTAGE_ALPHA_BASELINE}
You are Agent B: The Oracle.

Domain:
- Luck & Fate.
- External variables, historical accidents, inherited circumstances,
  geopolitical context, and sudden reversals.

Perspective:
- Identify what lies outside human control.
- Explain uncertainty, confidence limits, second-order effects, and fragility.
- Warn against plans that depend on favorable luck.

Output style:
- Realistic, contextual, probability-aware.
- Speak from the perspective of contingency and limits.
"""

oracle = Agent(
    name="The Oracle 🔮",
    model=make_model(ORACLE_MODEL),
    handoff_description=(
        "Specialist in Luck & Fate: contingency, uncertainty, historical accident, "
        "geopolitical reversals, and limits of planning."
    ),
    instructions=ORACLE_INSTRUCTIONS,
    hooks=RawToolArgumentsLoggerAgentHook(),
    model_settings=ModelSettings(
        temperature=ORACLE_TEMPERATURE,
        log_agent_calls=True,
        log_agent_handoffs=True,
    ),
)

# Survivor agent


In [ ]:
SURVIVOR_INSTRUCTIONS = f"""
{GREAT_OUTAGE_ALPHA_BASELINE}
You are Agent C: The Survivor.

Domain:
- Random Pain & Hardship.
- Crisis, suffering, war, betrayal, upheaval, systemic shocks.

Perspective:
- Analyze how hardship shapes legitimacy, endurance, trust, and resilience.
- Acknowledge human costs and the danger of cruelty or overreach.
- Distinguish necessary resilience measures from abusive measures.

Output style:
- Grounded, sober, resilience-oriented.
- Speak from the perspective of adversity and endurance.
"""

survivor = Agent(
    name="The Survivor ⚔️",
    model=make_model(SURVIVOR_MODEL),
    handoff_description=(
        "Specialist in Random Pain & Hardship: adversity, shocks, betrayal, "
        "war, systemic stress, and legitimacy under pressure."
    ),
    instructions=SURVIVOR_INSTRUCTIONS,
    hooks=RawToolArgumentsLoggerAgentHook(),
    model_settings=ModelSettings(
        temperature=SURVIVOR_TEMPERATURE,
        log_agent_calls=True,
        log_agent_handoffs=True,
    ),
)

# Prince Agent


In [ ]:
PRINCE_INSTRUCTIONS = f"""
{GREAT_OUTAGE_ALPHA_BASELINE}
You are Agent O: The Prince, the orchestrator of the Council.

Your role:
- You synthesize the perspectives of your Council consisting of:
    Agent A: The Architect: Work Ethic & Fortune.
    Agent B: The Oracle: Luck & Fate.
    Agent C: The Survivor: Random Pain & Hardship.

Venn domain logic for agent involvement:
- A ∩ B = Attitude & Knowledge:
  Strategic awareness of limits and opportunities; learning from history;
  knowing when effort matters and when restraint is wiser.

- A ∩ C = Character & Trustworthiness:
  Endurance under pressure; reputation through consistent conduct in adversity;
  legitimacy built through disciplined restraint.

- B ∩ C = Determination & Humour:
  Adaptability, resilience, timing, psychological flexibility, and responsiveness
  under uncertain hardship.

- A ∩ B ∩ C = Love & Destiny:
  Civic loyalty, legitimacy, tacit consent, and historically situated success.
  This is not mastery over fortune; it is alignment with fortune through
  disciplined agency.

Workflow:
First, identify the triggered domains and the relevant Venn intersection in the user query.
Then, consult the relevant specialists among veil, specter, and reflex according to the triggered Venn domains.
Make sure you have all the necessary informations to complete the request, or else come back to a specialist
to ask for more information.
Based on their advice, build a single coherent view to answer the query in the required format.
"""

prince = Agent(
    name="The Prince 👑",
    model=make_model(PRINCE_MODEL),
    instructions=PRINCE_INSTRUCTIONS,
    tools=[
        architect.as_tool(
            tool_name="consult_architect",
            tool_description=("Consult the Architect on preparation and institutions."),
        ),
        oracle.as_tool(
            tool_name="consult_oracle",
            tool_description=("Consult the Oracle on uncertainty and limits."),
        ),
        survivor.as_tool(
            tool_name="consult_survivor",
            tool_description=("Consult the Survivor on hardship and resilience."),
        ),
    ],
    model_settings=ModelSettings(
        temperature=PRINCE_TEMPERATURE,
        log_agent_calls=True,
        log_agent_handoffs=True,
    ),
)

# Setting logging parameters


In [ ]:
import logging, sys, time
from loguru import logger
from datetime import datetime

def get_timestamp() -> str:
    now = datetime.now()
    return now.strftime("%Y_%m_%d_%H_%M_%S_") + f"{now.microsecond:06d}"[:4]

logger.remove()
logger.add(sys.stderr, level="TRACE")
current_file_logger = None

def set_file_logger():
    global current_file_logger
    log_filename = f"logs/machiavel-agent-log-{get_timestamp()}.log"
    if current_file_logger != None:
        logger.remove(current_file_logger)
    current_file_logger = logger.add(log_filename, level="TRACE", rotation="500 MB")

def log_models_and_temperatures():
    logger.info(f"Architect Model: {ARCHITECT_MODEL}, Temperature: {ARCHITECT_TEMPERATURE}")
    logger.info(f"Oracle Model: {ORACLE_MODEL}, Temperature: {ORACLE_TEMPERATURE}")
    logger.info(f"Survivor Model: {SURVIVOR_MODEL}, Temperature: {SURVIVOR_TEMPERATURE}")
    logger.info(f"Prince Model: {PRINCE_MODEL}, Temperature: {PRINCE_TEMPERATURE}")

logging.getLogger("httpx").setLevel(logging.WARNING)
al = logging.getLogger("openai.agents")
al.setLevel(logging.DEBUG); al.propagate = False; al.filters.clear(); al.handlers.clear()
class F(logging.Filter): filter = lambda _, r: not any(x in r.getMessage() for x in ['No conversation_id', 'Tracing is disabled.', 'Setting current trace: no-op'])
class H(logging.Handler): emit = lambda self, r: (logger.info if "Running agent" in self.format(r) else logger.trace)(self.format(r))
al.addFilter(F()); al.addHandler(H())


# Running the Prince agent workflow


In [ ]:
import nest_asyncio
import asyncio

QUERY = f"""
# [USER QUERY]
{GREAT_OUTAGE_ALPHA_PROMPT}
"""

async def run_machiavellian_council(user_query: str) -> str:
    result = await Runner.run(
        prince,
        user_query,
        hooks=RawToolArgumentsLoggerRunHook(),
    )
    return result.final_output

TEMPS = [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]

for t in TEMPS:        
    ARCHITECT_TEMPERATURE = t
    ORACLE_TEMPERATURE = t
    SURVIVOR_TEMPERATURE = t
    PRINCE_TEMPERATURE = t
    set_file_logger()
    log_models_and_temperatures()
    nest_asyncio.apply()
    loop = asyncio.get_event_loop()
    output = loop.run_until_complete(run_machiavellian_council(QUERY))

    logger.info(f"Machiavel council output \n{output}")
    logger.success("Done !")


In [ ]:
print("\n# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #")
print("# # # # # # # #  Machiavellian Council Output # # # # # # # #")
print("# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #\n")

from IPython.display import display, Markdown
display(Markdown(output))
